# Prompt U-Net — GPU Memory Benchmark

Measures peak VRAM on one small and one large 3D volume using `pynvml` (same data as `nvidia-smi`).

**Size definitions** (matching `inference_speed.ipynb`):
- **Small**: ROI in-plane bbox extent ≤128 in both axes (P-UNet single-tile regime, no tiling overhead)
- **Large**: total_voxels > 192³ = 7,077,888 (nnInteractive AutoZoom trigger)

| Section | Content |
|---------|---------|
| **§1** | Scan NPZ test data — find representative small & large volumes |
| **§2** | GPU memory measurement via pynvml |
| **§3** | Results table |

---
## §1 — Find Representative Volumes

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path().resolve()
project_root = notebook_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
from data.test_data.ds_handler import load_dataset

In [ ]:
# Thresholds (matching inference_speed.ipynb)
PATCH_VOXELS = 192 ** 3  # 7,077,888  — Large threshold (nnInteractive AutoZoom)
TILE_LIMIT   = 128       # P-UNet single-tile limit (Small if ROI bbox ≤ this)

NPZ_PATHS = [
    project_root / 'data' / 'test_data' / 'FLARE_2022.npz',
    project_root / 'data' / 'test_data' / 'han_seg_ct.npz',
    project_root / 'data' / 'test_data' / 'han_seg_mri.npz',
    project_root / 'data' / 'test_data' / 'HCCTase_ceCT.npz',
    project_root / 'data' / 'test_data' / 'SegRap2023.npz',
    project_root / 'data' / 'test_data' / 'TotalSeg_mri.npz',
]

# ------------------------------------------------------------------
# Pre-index NPZ files: open once (mmap), build pid→idx and pid→shape
# ------------------------------------------------------------------
_NPZ = {}         # dataset_name → opened npz
_PID_IDX = {}     # (dataset_name, pid) → int index
_PID_SHAPE = {}   # (dataset_name, pid) → (D0, D1, D2)
_SEG_COUNT = {}   # (dataset_name, pid) → int

print('Indexing NPZ files ...')
for npz_path in NPZ_PATHS:
    ds_name = npz_path.stem
    data = np.load(str(npz_path), allow_pickle=False, mmap_mode='r')
    _NPZ[ds_name] = data
    for i, p in enumerate(data['_pids']):
        p = str(p)
        _PID_IDX[(ds_name, p)] = i
        _PID_SHAPE[(ds_name, p)] = data[f'{i}_image'].shape  # mmap shape, O(1), no data loaded
        _SEG_COUNT[(ds_name, p)] = int(data['_seg_counts'][i])
print(f'  {len(_PID_IDX)} patients indexed')

# ------------------------------------------------------------------
# Cached seg_labels loader (materialises once per pid, reused)
# ------------------------------------------------------------------
_SEG_CACHE = {}

def get_seg_labels(ds_name, pid):
    key = (ds_name, str(pid))
    if key in _SEG_CACHE:
        return _SEG_CACHE[key]
    data = _NPZ.get(ds_name)
    if data is None:
        return None
    i = _PID_IDX.get(key)
    if i is None:
        return None
    seg_count = _SEG_COUNT.get(key, 0)
    if seg_count == 1:
        seg_labels = np.asarray(data[f'{i}_seg_0']).astype(np.int32)
    else:
        seg_labels = np.zeros(_PID_SHAPE[key], dtype=np.int32)
        for j in range(seg_count):
            s = np.asarray(data[f'{i}_seg_{j}'])
            seg_labels[s != 0] = j + 1
    _SEG_CACHE[key] = seg_labels
    return seg_labels

# ------------------------------------------------------------------
# Vectorised ROI in-plane bbox + roi_slices — single bbox computation
# ------------------------------------------------------------------
def roi_bbox_and_slices(seg_labels, axis, roi):
    """Returns (max_bbox_h, max_bbox_w, roi_slices) for one ROI on one axis."""
    mask = np.moveaxis(seg_labels == roi, axis, 0)  # (S, H, W)
    S, H, W = mask.shape

    row_has_fg = np.any(mask, axis=2)  # (S, H)
    col_has_fg = np.any(mask, axis=1)  # (S, W)
    has_fg = np.any(row_has_fg, axis=1)  # (S,)

    first_row = np.argmax(row_has_fg, axis=1)
    last_row  = H - 1 - np.argmax(row_has_fg[:, ::-1], axis=1)
    first_col = np.argmax(col_has_fg, axis=1)
    last_col  = W - 1 - np.argmax(col_has_fg[:, ::-1], axis=1)

    bbox_h = np.where(has_fg, last_row - first_row + 1, 0)
    bbox_w = np.where(has_fg, last_col - first_col + 1, 0)
    roi_slices = int(has_fg.sum())
    return int(bbox_h.max()), int(bbox_w.max()), roi_slices

# ------------------------------------------------------------------
# Scan: for each (patient, axis) use np.bincount to find the largest ROI
# (by total voxel count) in O(n_voxels). Then compute bbox for that one ROI only.
# ~100× faster than iterating all 117+ ROIs per patient for multi-label datasets.
# ------------------------------------------------------------------
candidates = []
print('Scanning volumes (largest ROI per patient per axis) ...')
for npz_path in NPZ_PATHS:
    ds_name = npz_path.stem
    data = _NPZ[ds_name]
    for pid_idx, p in enumerate(data['_pids']):
        p = str(p)
        seg_labels = get_seg_labels(ds_name, p)
        if seg_labels is None:
            continue
        d0, d1, d2 = _PID_SHAPE[(ds_name, p)]
        img_shape = (d0, d1, d2)

        # Find the largest ROI (by total voxels) — single pass via bincount
        counts = np.bincount(seg_labels.ravel())
        if len(counts) <= 1:  # only background
            continue
        best_roi = int(np.argmax(counts[1:]) + 1)  # skip background (label 0)

        for axis in range(3):
            h, w = [img_shape[a] for a in range(3) if a != axis]
            max_h, max_w, roi_slices = roi_bbox_and_slices(seg_labels, axis, best_roi)
            if roi_slices == 0:
                continue
            total_voxels = roi_slices * h * w
            max_extent = max(max_h, max_w)
            candidates.append({
                'npz_path': str(npz_path), 'dataset_name': ds_name,
                'pid': p, 'axis': axis, 'roi': best_roi,
                'roi_slices': roi_slices,
                'h': h, 'w': w, 'total_voxels': total_voxels,
                'max_bbox_h': max_h, 'max_bbox_w': max_w,
                'max_bbox_extent': max_extent,
            })

# ---- Classify ----
def classify(c):
    s = c['max_bbox_extent'] <= TILE_LIMIT
    l = c['total_voxels'] > PATCH_VOXELS
    if s and l:   return 'Small & Large'
    if s:         return 'Small'
    if l:         return 'Large'
    return 'Medium'

for c in candidates:
    c['size_bin'] = classify(c)

from collections import Counter
print(f'Scanned {len(candidates)} (patient, axis) combinations (largest ROI only)')
print(f'Size distribution:')
for label, count in Counter(c['size_bin'] for c in candidates).most_common():
    print(f'  {label}: {count}')

In [ ]:
# Pick: largest Small by total_voxels (stress-test without tiling), largest Large overall
small_candidates = [c for c in candidates if c['size_bin'] == 'Small']
large_candidates = [c for c in candidates if c['size_bin'] == 'Large']

print(f'Small candidates: {len(small_candidates)}  |  Large candidates: {len(large_candidates)}')

rep_small = max(small_candidates, key=lambda c: c['total_voxels'])
rep_large = max(large_candidates, key=lambda c: c['total_voxels'])

for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    print(f"\n{label} volume:")
    print(f"  dataset         : {rep['dataset_name']}")
    print(f"  pid             : {rep['pid']}")
    print(f"  ROI             : {rep['roi']}")
    print(f"  axis            : {rep['axis']}")
    print(f"  roi_slices      : {rep['roi_slices']}")
    print(f"  in-plane        : {rep['h']}×{rep['w']}")
    print(f"  total_voxels    : {rep['total_voxels']:,}")
    print(f"  max_bbox_extent : {rep['max_bbox_extent']}  (≤128 = single tile)")

---
## §2 — GPU Memory Measurement

In [ ]:
import threading
import time

import tensorflow as tf
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo

from inference.inference_volume import VolumeInference
from inference.ssf import ConfidenceDropStrategy

nvmlInit()
_HANDLE = nvmlDeviceGetHandleByIndex(0)

def _gpu_used_mb():
    return nvmlDeviceGetMemoryInfo(_HANDLE).used / (1024 * 1024)


def measure_peak(inference_fn, poll_interval_s=0.1):
    """Run inference_fn while polling GPU used memory. Returns peak MB."""
    peak = [_gpu_used_mb()]
    stop = threading.Event()
    def poll():
        while not stop.is_set():
            cur = _gpu_used_mb()
            if cur > peak[0]:
                peak[0] = cur
            time.sleep(poll_interval_s)
    t = threading.Thread(target=poll, daemon=True)
    t.start()
    try:
        inference_fn()
    finally:
        stop.set()
        t.join()
    return peak[0]

# Init TF CUDA context and record baseline
_ = tf.zeros([1], dtype=tf.float32)
BASELINE_MB = _gpu_used_mb()
print(f'GPU baseline (idle + CUDA contexts): {BASELINE_MB:.1f} MB')

MODEL_PATH = project_root / 'training' / 'p_unet_332.keras'
print(f'Model: {MODEL_PATH}')

In [ ]:
def load_volume(rep):
    """Load a 3D volume and isolate the selected ROI."""
    dataset = load_dataset(rep['npz_path'])
    item = dataset[rep['pid']]
    img_3d = np.asarray(item['image']).astype(np.float32)
    segs = item['segmentations']
    if isinstance(segs, list):
        seg_labels = np.zeros_like(img_3d, dtype=np.int32)
        for li, s in enumerate(segs, 1):
            seg_labels[np.asarray(s) != 0] = li
    else:
        seg_labels = np.asarray(segs).astype(np.int32)
    seg_3d_binary = (seg_labels == rep['roi']).astype(np.float32)
    # Pick middle slice containing the ROI as prompt
    sum_axes = tuple(a for a in range(3) if a != rep['axis'])
    areas = seg_3d_binary.sum(axis=sum_axes)
    valid = np.where(areas > 0)[0]
    prompt_idx = valid[len(valid) // 2]
    prompt_2d = np.take(seg_3d_binary, prompt_idx, axis=rep['axis'])
    return img_3d, seg_3d_binary, prompt_2d, prompt_idx


def profile_volume(rep):
    """Run full VolumeInference and return peak VRAM above baseline."""
    img_3d, seg_3d_binary, prompt_2d, prompt_idx = load_volume(rep)
    vi = VolumeInference(
        model_path=str(MODEL_PATH),
        modality=rep['modality'],
        normalization='universal',
        ssf_strategy=ConfidenceDropStrategy(drop_fraction=0.05),
        buffer_size=4,
        batch_size=6,
    )
    # Warm-up — absorbs one-shot autotuning / JIT allocations
    _ = vi.run(img_3d=img_3d, seg_3d_binary=seg_3d_binary,
               initial_prompt_2d_seg=prompt_2d,
               prompt_axis=rep['axis'], prompt_idx=prompt_idx)
    # Measurement
    peak = measure_peak(lambda: vi.run(
        img_3d=img_3d, seg_3d_binary=seg_3d_binary,
        initial_prompt_2d_seg=prompt_2d,
        prompt_axis=rep['axis'], prompt_idx=prompt_idx,
    ))
    return peak - BASELINE_MB

In [ ]:
results = {}
for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    print(f'\nProfiling {label} volume: {rep["pid"]} (roi={rep["roi"]}, axis={rep["axis"]})')
    peak_mb = profile_volume(rep)
    results[label] = peak_mb
    print(f'  Peak VRAM (above baseline): {peak_mb:.1f} MB')
    # Clean up
    tf.keras.backend.clear_session()

---
## §3 — Results

In [ ]:
P_UNET_PARAMS = 28.0  # million
P_UNET_DISK   = 108   # MB (.keras)

print(f"{'='*66}")
print(f"  Prompt U-Net GPU Memory")
print(f"  Small = ROI in-plane bbox ≤ {TILE_LIMIT}   (P-UNet single-tile regime)")
print(f"  Large = total_voxels > {PATCH_VOXELS:,}  (nnInteractive AutoZoom)")
print(f"  Model: {P_UNET_PARAMS:.0f}M params, {P_UNET_DISK} MB on disk")
print(f"  Batch size: 6")
print(f"{'='*66}")
print(f"  {'Size':<8} {'Volume':<26} {'bbox_ext':<10} {'total_voxels':<16} {'Peak VRAM':<14}")
print(f"  {'-'*74}")
for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    name = f"{rep['dataset_name']}/{rep['pid']}"
    print(f"  {label:<8} {name:<26} {rep['max_bbox_extent']:<10} {rep['total_voxels']:<16,} {results[label]:.1f} MB")
print(f"{'='*66}")
print(f"\n  Reference (nnInteractive paper, RTX 4090):")
print(f"    Small  (≤192³) : < 6 GB")
print(f"    Large  (>192³) : < 10 GB")
print(f"  This measurement: RTX A6000 (48 GB)")